In [ ]:
# Cell 0 — Install & Import Library
!pip install -q huggingface_hub kaggle pillow

import os
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from huggingface_hub import HfApi, snapshot_download


In [ ]:
# Cell 1 — Konfigurasi & Kredensial (Inject dari Backend atau Kaggle Secrets)
HF_REPO_ID = os.environ.get("HF_REPO_ID", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
KAGGLE_DATASET_SLUG = os.environ.get("KAGGLE_DATASET_SLUG", "tokiva-pos-dataset")

MIN_IMAGE_DIM = 96
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Fallback ke Kaggle Secrets bila dijalankan manual di Kaggle Web UI
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    if not HF_TOKEN:
        try:
            HF_TOKEN = _secrets.get_secret("HF_TOKEN")
        except Exception:
            pass
    if not KAGGLE_KEY:
        try:
            KAGGLE_KEY = _secrets.get_secret("KAGGLE_KEY")
        except Exception:
            pass
    if not KAGGLE_USERNAME:
        try:
            KAGGLE_USERNAME = _secrets.get_secret("KAGGLE_USERNAME")
        except Exception:
            pass
except Exception:
    pass

_missing = [k for k, v in [("KAGGLE_USERNAME", KAGGLE_USERNAME), ("KAGGLE_KEY", KAGGLE_KEY), ("HF_REPO_ID", HF_REPO_ID)] if not v]
if _missing:
    raise RuntimeError(f"Konfigurasi/Kaggle Secrets berikut belum tersedia: {', '.join(_missing)}")

# Tulis kredensial Kaggle CLI (~/.kaggle/kaggle.json)
kaggle_cfg_dir = Path.home() / ".kaggle"
kaggle_cfg_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_path = kaggle_cfg_dir / "kaggle.json"
kaggle_json_path.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
try:
    os.chmod(kaggle_json_path, 0o600)
except Exception:
    pass

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY
print(f"Konfigurasi siap -> HF: {HF_REPO_ID} | Kaggle Dataset: {KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}")


In [ ]:
# Cell 2 — Ambil Commit HEAD Hugging Face & Unduh Snapshot Dataset Terbaru
hf_api = HfApi(token=HF_TOKEN if HF_TOKEN else None)
hf_commit_oid = "unknown"
try:
    commits = hf_api.list_repo_commits(repo_id=HF_REPO_ID, repo_type="dataset")
    if commits:
        hf_commit_oid = getattr(commits[0], "commit_id", None) or getattr(commits[0], "oid", None) or "unknown"
except Exception as exc:
    print(f"Peringatan cek commit HEAD HF: {exc}")

RAW_DIR = Path("/kaggle/working/hf_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mengunduh snapshot dari Hugging Face {HF_REPO_ID} (commit: {hf_commit_oid})...")
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN if HF_TOKEN else None,
    local_dir=str(RAW_DIR),
    ignore_patterns=["**/thumbnails/**", "**/.git/**"],
)
print("Snapshot Hugging Face selesai diunduh ke", RAW_DIR)


In [ ]:
# Cell 3 — Gabungkan data/ + dataset/, Validasi Gambar via PIL, dan Susun Staging Bersih
STAGING_DIR = Path("/kaggle/working/dataset_staging")
if STAGING_DIR.exists():
    shutil.rmtree(STAGING_DIR)
STAGING_DATA_DIR = STAGING_DIR / "data"
STAGING_DATA_DIR.mkdir(parents=True, exist_ok=True)

candidate_roots = [p for p in [RAW_DIR / "data", RAW_DIR / "dataset"] if p.exists() and p.is_dir()]
if not candidate_roots:
    candidate_roots = [RAW_DIR]

seen_files = set()
class_counts = {}
skipped_invalid = 0
total_images = 0

for root in candidate_roots:
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name.strip()
        if not class_name or class_name.startswith(".") or class_name.lower() == "thumbnails":
            continue
        if class_name in {"data", "dataset"}:
            continue

        dest_class_dir = STAGING_DATA_DIR / class_name
        for img_path in sorted(class_dir.glob("*")):
            if not img_path.is_file():
                continue
            if img_path.suffix.lower() not in VALID_EXTENSIONS:
                continue
            dedup_key = (class_name, img_path.name)
            if dedup_key in seen_files:
                continue

            try:
                with Image.open(img_path) as im:
                    w, h = im.size
                    im.load()
                if w < MIN_IMAGE_DIM or h < MIN_IMAGE_DIM:
                    skipped_invalid += 1
                    continue
            except Exception:
                skipped_invalid += 1
                continue

            dest_class_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest_class_dir / img_path.name)
            seen_files.add(dedup_key)
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
            total_images += 1

if total_images == 0:
    raise RuntimeError(f"Tidak ada citra valid (>= {MIN_IMAGE_DIM}x{MIN_IMAGE_DIM}) ditemukan dari {HF_REPO_ID}")

synced_at = datetime.now(timezone.utc).isoformat()
manifest = {
    "hf_repo": HF_REPO_ID,
    "hf_commit_oid": hf_commit_oid,
    "num_classes": len(class_counts),
    "num_images": total_images,
    "skipped_invalid": skipped_invalid,
    "classes": class_counts,
    "synced_at": synced_at,
}
(STAGING_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Staging siap: {len(class_counts)} kelas, {total_images} citra valid, {skipped_invalid} dilewati.")


In [ ]:
# Cell 4 — Buat / Perbarui Private Kaggle Dataset & Simpan sync_summary.json
dataset_id = f"{KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}"
dataset_metadata = {
    "title": "Tokiva POS Product Dataset",
    "id": dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(STAGING_DIR / "dataset-metadata.json").write_text(json.dumps(dataset_metadata, indent=2), encoding="utf-8")

check_res = subprocess.run(
    ["kaggle", "datasets", "status", dataset_id],
    capture_output=True,
    text=True,
)
dataset_exists = check_res.returncode == 0 and "404" not in (check_res.stdout + check_res.stderr)

short_oid = str(hf_commit_oid)[:8]
if dataset_exists:
    action = "updated"
    cmd = [
        "kaggle", "datasets", "version",
        "-p", str(STAGING_DIR),
        "-m", f"Sync HF {short_oid} ({len(class_counts)} kelas, {total_images} foto)",
        "--dir-mode", "zip",
    ]
else:
    action = "created"
    cmd = [
        "kaggle", "datasets", "create",
        "-p", str(STAGING_DIR),
        "--dir-mode", "zip",
    ]

push_res = subprocess.run(cmd, capture_output=True, text=True)
cli_output = (push_res.stdout or "") + ("\n" + push_res.stderr if push_res.stderr else "")
print(cli_output)
if push_res.returncode != 0:
    raise RuntimeError(f"Gagal {action} Kaggle Dataset {dataset_id} (code {push_res.returncode}): {cli_output}")

sync_summary = {
    "status": "complete",
    "action": action,
    "hf_repo": HF_REPO_ID,
    "hf_commit_oid": hf_commit_oid,
    "kaggle_dataset_id": dataset_id,
    "num_classes": len(class_counts),
    "num_images": total_images,
    "skipped_invalid": skipped_invalid,
    "classes": class_counts,
    "synced_at": synced_at,
    "cli_output": cli_output.strip(),
}
Path("/kaggle/working/sync_summary.json").write_text(json.dumps(sync_summary, indent=2), encoding="utf-8")
print(f"Selesai ({action}): {dataset_id} -> {len(class_counts)} kelas, {total_images} foto.")
